In [1]:
import csv
import json
import re
from pathlib import Path
from types import SimpleNamespace
from typing import Dict, List, Optional, Tuple, TypedDict, Union, Any

## Utility di parsing dose (generiche, condivise da tutti gli score)

In [2]:
# Il target di un farmaco puo' essere un valore singolo (dose totale
# giornaliera) oppure un intervallo (min, max) di dose totale giornaliera.
TargetDose = Union[float, Tuple[float, float]]

def _parse_dose_value(active_ingredient: Optional[str]) -> Optional[float]:
    """
    Estrae il valore numerico della dose (per singola unità) dal campo
    `active_ingredient` dello schema (es. "20 mg" -> 20.0, "12,5 mg" -> 12.5).
 
    Per farmaci in combinazione (es. "97/103 mg" o "12,5/25 mg") restituisce 
    la somma dei due valori (es. sacubitril/valsartan).
    """
    if not active_ingredient:
        return None

    # 1. Aggiunta della virgola nella regex: [\d.,/]
    match = re.match(r"([\d.,/]+)\s*([a-zA-Zµ]+)", active_ingredient.strip())
    if not match:
        return None

    # 2. Normalizzazione della virgola in punto
    numbers_part = match.group(1).replace(",", ".")
    
    try:
        numbers = [float(n) for n in numbers_part.split("/") if n]
    except ValueError:
        return None

    return sum(numbers) if numbers else None
 
 
def _compute_daily_dose(entry) -> Optional[float]:
    """
    Calcola la dose totale giornaliera assunta dal paziente per un dato
    TherapyEntry, moltiplicando la dose per unità (active_ingredient) per
    il numero di unità assunte al giorno (dose_units_number).
    """
    dose_per_unit = _parse_dose_value(getattr(entry, "active_ingredient", None))
    dose_units_number = getattr(entry, "dose_units_number", None)
 
    if dose_per_unit is None or dose_units_number is None:
        return None
 
    return dose_per_unit * dose_units_number
 
 
def _get_drug_class_value(entry) -> Optional[str]:
    """Restituisce il valore stringa di drug_class, gestendo sia Enum che str."""
    drug_class = getattr(entry, "drug_class", None)
    if drug_class is None:
        return None
    return drug_class.value if hasattr(drug_class, "value") else str(drug_class)
 
 
# Range: "10–20 mg", "2.5-5 mg" (riconosce diversi tipi di trattino).
_RANGE_RE = re.compile(r"^([\d.]+)\s*[–—−-]\s*([\d.]+)\s*([a-zA-Zµ]+)$")
 
# Combinazione a unità NON condivisa, es. "37.5 mg/20 mg" (a differenza di
# "97/103 mg", ogni componente ha la propria unità ripetuta).
_COMBO_SEPARATE_UNITS_RE = re.compile(
    r"^([\d.]+)\s*([a-zA-Zµ]+)\s*/\s*([\d.]+)\s*([a-zA-Zµ]+)$"
)
 
def _parse_target_dose_string(dose_str: Optional[str]) -> Optional[TargetDose]:
    """
    Estrae la dose target per singola somministrazione da una stringa della
    tabella delle linee guida (colonna `target_dose`). Va poi moltiplicata
    per `target_dose_daily_intakes` per ottenere la dose totale giornaliera
    (vedi `load_target_doses_...`).

    Gestisce tre notazioni:
    - valore singolo: "6.25 mg" -> 6.25
    - combinazione a unità condivisa: "97/103 mg" -> 200.0 (somma)
    - combinazione a unità ripetute: "37.5 mg/20 mg" -> 57.5 (somma, solo
      se le due unità coincidono; altrimenti non sommabile in modo
      affidabile -> None)
    - range: "10–20 mg" -> (10.0, 20.0)  (tupla, NON sommata)
    """
    if not dose_str:
        return None
    
    # Normalizzazione della virgola in punto e rimozione spazi di testa/coda
    dose_str = dose_str.strip().replace(",", ".")

    range_match = _RANGE_RE.match(dose_str)
    if range_match:
        low, high, _unit = range_match.groups()
        return (float(low), float(high))

    combo_match = _COMBO_SEPARATE_UNITS_RE.match(dose_str)
    if combo_match:
        n1, u1, n2, u2 = combo_match.groups()
        if u1.lower() != u2.lower():
            return None
        return float(n1) + float(n2)

    # Formato semplice o combinazione a unità condivisa (es. "97/103 mg"):
    # riutilizza lo stesso parser usato lato paziente.
    return _parse_dose_value(dose_str)
 
 
def _percent_of_target(
    daily_dose: Optional[float],
    target_dose: Optional[TargetDose],
) -> Optional[float]:
    """
    Calcola la percentuale di dose target raggiunta, a partire dalla dose
    giornaliera totale del paziente e dal target (caricato da
    `load_target_doses_...`).
 
    `target_dose` può essere:
    - un valore singolo (float): confronto diretto, daily_dose/target*100.
    - un intervallo (min, max): se la dose del paziente rientra
      nell'intervallo, la dose target è considerata raggiunta (100%); se è
      sotto il limite inferiore, la percentuale è calcolata rispetto al
      limite inferiore (quindi <100%); se supera il limite superiore, la
      percentuale è calcolata rispetto al limite superiore (quindi >100%,
      comunque trattata come target raggiunto/superato dalle funzioni di
      scoring, che limitano il punteggio al massimo previsto, anche se non
      dovrebbe succedere).
    """
    if daily_dose is None or target_dose is None:
        return None
 
    if isinstance(target_dose, tuple):
        low, high = target_dose
        if low <= daily_dose <= high:
            return 100.0
        if daily_dose < low:
            return (daily_dose / low) * 100 if low else None
        return (daily_dose / high) * 100 if high else None
 
    if not target_dose:
        return None
    return (daily_dose / target_dose) * 100
 
 
def load_target_doses_from_json(path) -> Dict[str, TargetDose]:
    """
    Carica le dosi target dal file `HFrEF_medication_table.json` (struttura
    annidata: lista di blocchi -> `medications` -> `drugs`).
 
    Restituisce {generic_drug_name in minuscolo: dose_target_giornaliera},
    dove il valore è la dose totale/die (target_dose per singola
    somministrazione × target_dose_daily_intakes), oppure una tupla
    (min, max) se la dose in linea guida è espressa come range.
 
    Farmaci la cui dose non è interpretabile (formato non gestito) vengono 
    esclusi dal dizionario.
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
 
    target_doses: Dict[str, TargetDose] = {}
    for block in data:
        for med_class in block.get("medications", []):
            for drug in med_class.get("drugs", []):
                generic_name = (drug.get("generic_drug_name") or "").strip().lower()
                if not generic_name:
                    continue
 
                per_intake_dose = _parse_target_dose_string(drug.get("target_dose"))
                if per_intake_dose is None:
                    continue
 
                intakes = drug.get("target_dose_daily_intakes") or 1
                if isinstance(per_intake_dose, tuple):
                    target_doses[generic_name] = (
                        per_intake_dose[0] * intakes,
                        per_intake_dose[1] * intakes,
                    )
                else:
                    target_doses[generic_name] = per_intake_dose * intakes
 
    return target_doses
 
 
def load_target_doses_from_csv(path) -> Dict[str, TargetDose]:
    """
    Carica le dosi target dal file `HFrEF_medication_table.csv` (formato
    "piatto": una riga per farmaco, colonne generic_drug_name, target_dose,
    target_dose_daily_intakes, ecc.).
 
    Stesso comportamento/formato di ritorno di `load_target_doses_from_json`.
    """
    target_doses: Dict[str, TargetDose] = {}
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            generic_name = (row.get("generic_drug_name") or "").strip().lower()
            if not generic_name:
                continue
 
            per_intake_dose = _parse_target_dose_string(row.get("target_dose"))
            if per_intake_dose is None:
                continue
 
            try:
                intakes = int(row.get("target_dose_daily_intakes") or 1)
            except ValueError:
                intakes = 1
 
            if isinstance(per_intake_dose, tuple):
                target_doses[generic_name] = (
                    per_intake_dose[0] * intakes,
                    per_intake_dose[1] * intakes,
                )
            else:
                target_doses[generic_name] = per_intake_dose * intakes
 
    return target_doses
 
 
def load_target_doses(path) -> Dict[str, TargetDose]:
    """
    Dispatcher: carica le dosi target dal file delle linee guida,
    scegliendo automaticamente il parser in base all'estensione
    (`.json` o `.csv`). Restituisce il dizionario `target_doses` da passare
    a `calculate_mhfc_score`,`calculate_ops_score`,`calculate_c4p_score` o 
    `calculate_adherence_scores`.
    """
    path_str = str(path).lower()
    if path_str.endswith(".json"):
        return load_target_doses_from_json(path)
    if path_str.endswith(".csv"):
        return load_target_doses_from_csv(path)
    raise ValueError(f"Formato non supportato per il file delle dosi target: {path}")

In [3]:
# Match più stringente
def _lookup_target_dose(
    generic_name: str,
    target_doses: Dict[str, TargetDose],
) -> Optional[TargetDose]:
    """
    Cerca la dose target di un farmaco a partire dal suo generic_drug_name
    (già in minuscolo). Prova prima il match ESATTO con le chiavi di
    `target_doses`; se non trova nulla, cerca un match "per contenimento"
    in entrambe le direzioni (una stringa contenuta nell'altra) — utile
    quando il nome estratto dalla lettera clinica resta in una lingua
    diversa da quella della linea guida (es. "bisoprololo" vs
    "bisoprolol", "furosemide" vs "furosemide 25 mg" ecc.).

    In caso di più match per contenimento, viene restituito quello con la
    sovrapposizione più lunga (il nome candidato più simile), per ridurre
    il rischio di falsi positivi su nomi molto corti.
    """
    if not generic_name:
        return None

    if generic_name in target_doses:
        return target_doses[generic_name]

    best_match_key = None
    best_match_len = 0
    for key in target_doses:
        if key in generic_name or generic_name in key:
            overlap_len = min(len(key), len(generic_name))
            if overlap_len > best_match_len:
                best_match_key = key
                best_match_len = overlap_len

    return target_doses[best_match_key] if best_match_key is not None else None

In [4]:
# Match meno stringente (con punteggio e filtri per ridurre falsi positivi)
def _lookup_target_dose(
    generic_name: str,
    target_doses: Dict[str, Any],
) -> Optional[Any]:
    """
    Cerca la dose target di un farmaco a partire dal suo generic_drug_name.
    
    Scompone le stringhe in singoli termini (token) e confronta i termini
    tra loro anche per contenimento/prefisso (es. "metoprolol" vs "metoprololo"),
    applicando un sistema di punteggio e filtri per evitare falsi positivi.
    """
    if not generic_name:
        return None

    generic_clean = generic_name.strip().lower()

    # 1. Match Esatto diretto (il più veloce e sicuro)
    if generic_clean in target_doses:
        return target_doses[generic_clean]

    # Helper per estrarre parole significative (escludendo punteggiatura e parole < 4 lettere)
    def _get_tokens(text: str) -> set[str]:
        return {w for w in re.findall(r"\b[a-z0-9]+", text.lower()) if len(w) >= 4}

    input_tokens = _get_tokens(generic_clean)
    if not input_tokens:
        return None

    best_match_key = None
    best_score = 0.0

    for key in target_doses:
        key_tokens = _get_tokens(key)
        if not key_tokens:
            continue

        score = 0.0
        for t1 in input_tokens:
            for t2 in key_tokens:
                # Match esatto tra singoli termini (es. "metoprolol" == "metoprolol")
                if t1 == t2:
                    score += 1.0
                # Match per contenimento/prefisso (es. "metoprolol" in "metoprololo")
                # Richiede almeno 5 caratteri per evitare falsi positivi su suffissi corti
                elif (t1 in t2 or t2 in t1) and min(len(t1), len(t2)) >= 5:
                    overlap_ratio = min(len(t1), len(t2)) / max(len(t1), len(t2))
                    score += 0.8 * overlap_ratio

        if score > best_score:
            best_score = score
            best_match_key = key

    # Soglia di sicurezza: accetta il match solo con punteggio d'affinità sufficiente
    if best_match_key is not None and best_score >= 0.7:
        return target_doses[best_match_key]

    return None

## Tipi condivisi

In [5]:
class DrugScoreDetail(TypedDict):
    drug_name: Optional[str]
    drug_class: Optional[str]
    percent_of_target: Optional[float]
    points: float
    note: Optional[str]

class ScoreResult(TypedDict):
    total_score: float
    max_score: int
    breakdown: List[DrugScoreDetail]

## Funzioni di calcolo degli Scores

### mHFC 2022

In [6]:
def calculate_mhfc_score(
    therapy_list: Optional[List],
    target_doses: Dict[str, float],
    extended: bool = False,
) -> ScoreResult:
    """
    Parameters
    ----------
    therapy_list : List[TherapyEntry] | None
        Lista dei farmaci estratti dalla lettera clinica (`patient.discharge_therapy`).
    target_doses : Dict[str, float]
        Dizionario {generic_drug_name in minuscolo: dose_target_giornaliera}.
        Popolato a partire dalle tabelle delle dosi target della linea guida.
    extended : bool, default False
        Se True calcola lo score esteso (0-12) includendo anche
        idralazina-nitrato, ivabradina e vericiguat.
        Se False calcola lo score classico (0-9).
 
    Returns
    -------
    ScoreResult
        Dizionario con:
        - total_score: punteggio totale calcolato
        - max_score: punteggio massimo teorico (9 o 12)
        - breakdown: dettaglio del punteggio assegnato a ciascun farmaco
    """
    # -- Configurazione specifica di questo score (variabili locali) --------

    # Classi il cui punteggio dipende dalla percentuale di dose target
    # raggiunta. Uso i .value dell'enum Drug_Class (non i nomi) perche'
    # _get_drug_class_value restituisce sempre la stringa "value".
    dose_dependent_classes = {"ACE-I", "ARB", "Beta-blocker"}
 
    # Classi con punteggio fisso, indipendente dalla dose
    flat_score_classes = {
        "ARNI": 3.0,
        "MRA": 2.0,
        "SGLT2 Inhibitor": 2.0,
    }
 
    # Classe enum usata per idralazina-nitrato / ivabradina / vericiguat,
    # dato che Drug_Class non prevede valori dedicati per questi
    # farmaci: vengono estratti come "Other" e riconosciuti qui per nome.
    other_class_value = "Other"
 
    # Farmaci aggiuntivi considerati solo nello score esteso (0-12).
    # Il matching avviene per sottostringa su drug_name / generic_drug_name.
    # NOTA: idralazina e nitrato (isosorbide) sono un'unica riga della
    # tabella (terapia di combinazione "idralazina-nitrato" = 1 punto
    # complessivo), quindi condividono lo stesso class_key per evitare di
    # contarli due volte.
    extended_drugs = {
        "hydralazine": ("HYDRALAZINE_NITRATE", 1.0),
        "isosorbide": ("HYDRALAZINE_NITRATE", 1.0),   # idralazina-nitrato (combo)
        "ivabradine": ("IVABRADINE", 1.0),
        "vericiguat": ("VERICIGUAT", 1.0),
    }
 
    max_score = 12 if extended else 9
 
    if not therapy_list:
        return {"total_score": 0.0, "max_score": max_score, "breakdown": []}
 
    # Punteggio massimo raggiunto per ciascuna "classe" ai fini dello score,
    # cosi' da non sommare piu' volte la stessa classe se il paziente assume
    # piu' farmaci equivalenti (es. due beta-bloccanti diversi).
    class_scores: Dict[str, float] = {}
    breakdown: List[DrugScoreDetail] = []
 
    for entry in therapy_list:
        drug_class_value = _get_drug_class_value(entry)
        drug_name = getattr(entry, "drug_name", None)
        generic_name = (getattr(entry, "generic_drug_name", None) or "").strip().lower()
 
        points = 0.0
        percent_of_target: Optional[float] = None
        note: Optional[str] = None
        class_key: Optional[str] = None
 
        if drug_class_value in dose_dependent_classes:
            # ACEi e ARB condividono la stessa "riga" della tabella e sono
            # comunque mutuamente esclusivi rispetto ad ARNI: li accorpo
            # in un'unica classe ai fini del punteggio.
            class_key = "ACEI_ARB" if drug_class_value in ("ACE-I", "ARB") else drug_class_value
 
            daily_dose = _compute_daily_dose(entry)
            target_dose = _lookup_target_dose(generic_name, target_doses)
            percent_of_target = _percent_of_target(daily_dose, target_dose)
 
            if percent_of_target is not None:
                if percent_of_target >= 50:
                    points = 2.0
                elif percent_of_target > 0:
                    points = 1.0
                else:
                    points = 0.0
            else:
                # Dose target non ancora disponibile per questo farmaco:
                # si assegna il punteggio minimo (farmaco presente in
                # terapia) segnalandolo in nota, in attesa dell'integrazione
                # con le tabelle delle dosi target.
                points = 1.0
                note = "Dose target non disponibile: assegnato punteggio minimo (1)."
 
        elif drug_class_value in flat_score_classes:
            class_key = drug_class_value
            points = flat_score_classes[drug_class_value]
 
        elif extended and drug_class_value in (other_class_value, None):
            # Idralazina-nitrato, ivabradina e vericiguat non hanno una
            # Drug_Class dedicata: vengono estratti come "Other" e qui
            # riconosciuti per nome (generic_drug_name o drug_name).
            search_text = f"{generic_name} {(drug_name or '').lower()}"
            matched = next(
                ((key, val) for key, val in extended_drugs.items() if key in search_text),
                None,
            )
            if matched is None:
                continue
            class_key, points = matched[1]  # (class_key, punti)
 
        else:
            # Farmaco non rilevante ai fini dello score classico
            # (es. loop diuretico, e idralazina/ivabradina/vericiguat
            # quando extended=False).
            continue
 
        # Tiene solo il punteggio massimo ottenuto per la classe
        if class_key not in class_scores or points > class_scores[class_key]:
            class_scores[class_key] = points
 
        breakdown.append({
            "drug_name": drug_name,
            "drug_class": drug_class_value,
            "percent_of_target": round(percent_of_target, 1) if percent_of_target is not None else None,
            "points": points,
            "note": note,
        })
 
    total_score = sum(class_scores.values())
 
    return {
        "total_score": total_score,
        "max_score": max_score,
        "breakdown": breakdown,
    }

### OPS

In [7]:
def calculate_ops_score(
    therapy_list: Optional[List],
    target_doses: Dict[str, float],
) -> ScoreResult:
    """
    Stessa logica generale di `calculate_mhfc_score`, ma con regole di
    punteggio proprie:
    - ACEi/ARB, ARNI, Beta-bloccanti, MRA: dose-dipendenti, con soglia al
      100% della dose target (sub-target vs dose target), non al 50%.
    - ARNI: punteggio piu' alto (2/4) rispetto ad ACEi/ARB (1/2).
    - SGLT2i: punteggio fisso (2 punti) per sola presenza del farmaco.
    - Nessuna voce per idralazina-nitrato/ivabradina/vericiguat.
 
    Parameters
    ----------
    therapy_list : List[TherapyEntry] | None
        Lista dei farmaci estratti dalla lettera clinica, tipicamente
        `patient.current_therapy` oppure `patient.discharge_therapy`.
    target_doses : Dict[str, float]
        Dizionario {generic_drug_name in minuscolo: dose_target_giornaliera},
        lo stesso usato da `calculate_mhfc_score`.
 
    Returns
    -------
    ScoreResult
        Dizionario con total_score, max_score (10) e breakdown per farmaco.
    """
    # -- Configurazione specifica di questo score (variabili locali) --------
 
    # Classi dose-dipendenti per OPS, con punteggio (sub_target, dose_target).
    # A differenza di mHFC, qui la soglia e' al 100% della dose target (non
    # al 50%) e anche MRA e' dose-dipendente.
    dose_dependent_points = {
        "ACE-I": (1.0, 2.0),
        "ARB": (1.0, 2.0),
        "ARNI": (2.0, 4.0),
        "Beta-blocker": (1.0, 2.0),
        "MRA": (1.0, 2.0),
    }
 
    # Classi con punteggio fisso, assegnato per sola presenza del farmaco
    flat_score_classes = {
        "SGLT2 Inhibitor": 2.0,
    }
 
    max_score = 10  # 4 (ARNI a target) + 2 (BB) + 2 (MRA) + 2 (SGLT2i)
 
    if not therapy_list:
        return {"total_score": 0.0, "max_score": max_score, "breakdown": []}
 
    class_scores: Dict[str, float] = {}
    breakdown: List[DrugScoreDetail] = []
 
    for entry in therapy_list:
        drug_class_value = _get_drug_class_value(entry)
        drug_name = getattr(entry, "drug_name", None)
        generic_name = (getattr(entry, "generic_drug_name", None) or "").strip().lower()
 
        points = 0.0
        percent_of_target: Optional[float] = None
        note: Optional[str] = None
        class_key: Optional[str] = None
 
        if drug_class_value in dose_dependent_points:
            sub_target_points, dose_target_points = dose_dependent_points[drug_class_value]
 
            # ACEi e ARB condividono la stessa riga della tabella (e sono
            # mutuamente esclusivi rispetto ad ARNI): li accorpo in
            # un'unica classe ai fini del punteggio.
            class_key = "ACEI_ARB" if drug_class_value in ("ACE-I", "ARB") else drug_class_value
 
            daily_dose = _compute_daily_dose(entry)
            target_dose = _lookup_target_dose(generic_name, target_doses)
            percent_of_target = _percent_of_target(daily_dose, target_dose)
 
            if percent_of_target is not None:
                if percent_of_target >= 100:
                    points = dose_target_points
                elif percent_of_target > 0:
                    points = sub_target_points
                else:
                    points = 0.0
            else:
                # Dose target non ancora disponibile: si assegna il
                # punteggio minimo (sub-target), segnalandolo in nota.
                points = sub_target_points
                note = "Dose target non disponibile: assegnato punteggio minimo (sub-target)."
 
        elif drug_class_value in flat_score_classes:
            class_key = drug_class_value
            points = flat_score_classes[drug_class_value]
 
        else:
            # Farmaco non previsto dalla tabella OPS (es. loop diuretico,
            # idralazina-nitrato, ivabradina, vericiguat).
            continue
 
        if class_key not in class_scores or points > class_scores[class_key]:
            class_scores[class_key] = points
 
        breakdown.append({
            "drug_name": drug_name,
            "drug_class": drug_class_value,
            "percent_of_target": round(percent_of_target, 1) if percent_of_target is not None else None,
            "points": points,
            "note": note,
        })
 
    total_score = sum(class_scores.values())
 
    return {
        "total_score": total_score,
        "max_score": max_score,
        "breakdown": breakdown,
    }

### Modified C4P

In [8]:
def calculate_c4p_score(
    therapy_list: Optional[List],
    target_doses: Dict[str, float],
) -> ScoreResult:
    """
    Per ciascuna delle classi ACEi/ARB, Beta-bloccanti, MRA e SGLT2i, il
    contributo è calcolato come:
 
        2 + 2 × (% dose target)   (frazione 0-1), fino a un massimo di 4
 
    Per ARNI, il contributo è calcolato come:
 
        4 + 2 × (% dose target)   (frazione 0-1), fino a un massimo di 6
 
    Se un farmaco di una data classe non è presente in terapia, il suo
    contributo è 0.
 
    Parameters
    ----------
    therapy_list : List[TherapyEntry] | None
        Lista dei farmaci estratti dalla lettera clinica, tipicamente
        `patient.current_therapy` oppure `patient.discharge_therapy`.
    target_doses : Dict[str, float]
        Dizionario {generic_drug_name in minuscolo: dose_target_giornaliera},
        lo stesso usato da `calculate_mhfc_score` / `calculate_ops_score`.
 
    Returns
    -------
    ScoreResult
        Dizionario con total_score, max_score (18) e breakdown per farmaco.
    """
    # -- Configurazione specifica di questo score (variabili locali) --------
 
    # Per ciascuna classe: (valore base, coefficiente moltiplicativo,
    # punteggio massimo). Formula: base + coefficiente * frazione_dose,
    # con frazione_dose limitata all'intervallo [0, 1].
    dose_formula_classes = {
        "ACE-I": (2.0, 2.0, 4.0),
        "ARB": (2.0, 2.0, 4.0),
        "Beta-blocker": (2.0, 2.0, 4.0),
        "MRA": (2.0, 2.0, 4.0),
        "SGLT2 Inhibitor": (2.0, 2.0, 4.0),
        "ARNI": (4.0, 2.0, 6.0),
    }
 
    max_score = 18  # 6 (ARNI a target) + 4 (BB) + 4 (MRA) + 4 (SGLT2i)
 
    if not therapy_list:
        return {"total_score": 0.0, "max_score": max_score, "breakdown": []}
 
    class_scores: Dict[str, float] = {}
    breakdown: List[DrugScoreDetail] = []
 
    for entry in therapy_list:
        drug_class_value = _get_drug_class_value(entry)
        drug_name = getattr(entry, "drug_name", None)
        generic_name = (getattr(entry, "generic_drug_name", None) or "").strip().lower()
 
        if drug_class_value not in dose_formula_classes:
            # Farmaco non previsto dalla formula C4P (es. loop diuretico,
            # idralazina-nitrato, ivabradina, vericiguat).
            continue
 
        base, coefficient, class_max = dose_formula_classes[drug_class_value]
 
        # ACEi e ARB condividono la stessa riga della formula (e sono
        # mutuamente esclusivi rispetto ad ARNI): li accorpo in un'unica
        # classe ai fini del punteggio, come negli altri score.
        class_key = "ACEI_ARB" if drug_class_value in ("ACE-I", "ARB") else drug_class_value
 
        daily_dose = _compute_daily_dose(entry)
        target_dose = _lookup_target_dose(generic_name, target_doses)
        percent_of_target = _percent_of_target(daily_dose, target_dose)
        note: Optional[str] = None
 
        if percent_of_target is not None:
            # La frazione usata nella formula e' limitata a [0, 1]: una
            # dose superiore al target non fa superare il punteggio massimo
            # della classe.
            dose_fraction = max(0.0, min(percent_of_target / 100, 1.0))
            points = base + coefficient * dose_fraction
        else:
            # Dose target non ancora disponibile: il farmaco e' comunque
            # presente in terapia, quindi si assegna il valore base della
            # formula, segnalandolo in nota.
            points = base
            note = f"Dose target non disponibile: assegnato punteggio base ({base})."
 
        points = min(points, class_max)
 
        if class_key not in class_scores or points > class_scores[class_key]:
            class_scores[class_key] = points
 
        breakdown.append({
            "drug_name": drug_name,
            "drug_class": drug_class_value,
            "percent_of_target": round(percent_of_target, 1) if percent_of_target is not None else None,
            "points": points,
            "note": note,
        })
 
    total_score = sum(class_scores.values())
 
    return {
        "total_score": total_score,
        "max_score": max_score,
        "breakdown": breakdown,
    }

### Calcolo combinato

In [9]:
def calculate_adherence_scores(
    therapy_list: Optional[List],
    target_doses: Dict[str, float],
    extended_mhfc: bool = False,
) -> Dict[str, ScoreResult]:
    """
    Calcola in un'unica chiamata gli score mHFC, OPS e modified C4P,
    a partire dalla stessa lista di farmaci e dallo stesso dizionario di
    dosi target.
 
    Returns
    -------
    dict con chiavi 'mhfc', 'ops' e 'c4p', ciascuna nel formato restituito
    dalla rispettiva funzione di scoring.
    """
    return {
        "mhfc": calculate_mhfc_score(therapy_list, target_doses, extended=extended_mhfc),
        "ops": calculate_ops_score(therapy_list, target_doses),
        "c4p": calculate_c4p_score(therapy_list, target_doses),
    }

## Calcolo degli scores su una lettera clinica

### Fz.ni di lettura

In [10]:
def load_clinical_letter(path) -> dict:
    """
    Carica il JSON prodotto dalla pipeline di information extraction per una
    singola lettera clinica.
    """
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def _dicts_to_therapy_entries(therapy_list: Optional[List[dict]]) -> List:
    """
    Converte una lista di dict (i TherapyEntry così come compaiono nel JSON
    esportato dalla pipeline di IE) in oggetti con accesso ad attributo
    (SimpleNamespace).
    """
    if not therapy_list:
        return []
    return [SimpleNamespace(**entry) for entry in therapy_list]

### Fz.ni di presentazione dei risultati

In [11]:
_SCORE_LABELS = {"mhfc": "mHFC 2022", "ops": "OPS", "c4p": "Modified C4P"}
 
 
def print_scores_summary(scores: Dict[str, ScoreResult], title: str = "") -> None:
    """
    Stampa a schermo una tabella riassuntiva (punteggio totale, massimo
    teorico, percentuale) per i tre score restituiti da
    `calculate_adherence_scores`.
    """
    if title:
        print(f"\n=== {title} ===")
 
    header = f"{'Score':<15}{'Punteggio':>12}{'Massimo':>10}{'%':>9}"
    print(header)
    print("-" * len(header))
    for key, label in _SCORE_LABELS.items():
        result = scores.get(key)
        if not result:
            continue
        total = result["total_score"]
        max_score = result["max_score"]
        percent = (total / max_score * 100) if max_score else 0.0
        print(f"{label:<15}{total:>12.1f}{max_score:>10}{percent:>8.1f}%")
 
 
def print_scores_breakdown(scores: Dict[str, ScoreResult]) -> None:
    """
    Stampa a schermo una tabella che confronta, farmaco per farmaco, i
    punti assegnati da ciascuno dei tre score (mHFC, OPS, C4P) — utile per
    capire a colpo d'occhio da dove derivi la differenza tra i punteggi
    totali.
    """
    points_by_drug: Dict[str, Dict[str, float]] = {}
    drug_order: List[str] = []
 
    for score_key in _SCORE_LABELS:
        for item in scores.get(score_key, {}).get("breakdown", []):
            name = item["drug_name"] or item["drug_class"] or "?"
            if name not in points_by_drug:
                points_by_drug[name] = {}
                drug_order.append(name)
            points_by_drug[name][score_key] = item["points"]
 
    def _fmt(value: Optional[float]) -> str:
        return f"{value:.1f}" if value is not None else "-"
 
    header = f"{'Farmaco':<32}{'mHFC':>8}{'OPS':>8}{'C4P':>8}"
    print(header)
    print("-" * len(header))
    for name in drug_order:
        row = points_by_drug[name]
        print(
            f"{name:<32}"
            f"{_fmt(row.get('mhfc')):>8}"
            f"{_fmt(row.get('ops')):>8}"
            f"{_fmt(row.get('c4p')):>8}"
        )

In [ ]:
if __name__ == "__main__":
    CLINICAL_LETTER_PATH = f"HFrEF_results/pz_1_cardiologia_gpt-oss_120b-cloud_with_span.json"
    GUIDELINE_TABLE_PATH = f"data/HFrEF_medication_table.json"

    target_doses = load_target_doses(GUIDELINE_TABLE_PATH)
    patient_data = load_clinical_letter(CLINICAL_LETTER_PATH)
 
    current_therapy = _dicts_to_therapy_entries(patient_data.get("current_therapy"))
    discharge_therapy = _dicts_to_therapy_entries(patient_data.get("discharge_therapy"))
 
    scores_current = calculate_adherence_scores(current_therapy, target_doses)
    scores_discharge = calculate_adherence_scores(discharge_therapy, target_doses)
 
    print_scores_summary(scores_current, title="Terapia attuale (current_therapy)")
    print_scores_breakdown(scores_current)
 
    print_scores_summary(scores_discharge, title="Terapia alla dimissione (discharge_therapy)")
    print_scores_breakdown(scores_discharge)

## Calcolo degli scores su una lettera per tutti i modelli

In [14]:
_MODEL_FILENAME_RE = re.compile(r"^pz_\d+_cardiologia_(.+)_with_span\.json$")

def _extract_model_name(filename: str) -> str:
    """Estrae il nome del modello dal nome del file, secondo la convenzione
    'pz_<n>_cardiologia_<nome_modello>_with_span.json'. Se il file non
    rispetta la convenzione, restituisce il nome del file così com'è."""
    match = _MODEL_FILENAME_RE.match(filename)
    return match.group(1) if match else filename


def compare_models_adherence_scores(
    patient_id,
    results_dir,
    target_doses: Dict[str, TargetDose],
    therapy_field: str = "discharge_therapy",
    extended_mhfc: bool = False,
) -> Dict[str, Dict[str, ScoreResult]]:
    """
    Calcola gli score di aderenza (mHFC, OPS, C4P) per lo stesso paziente a
    partire dalle estrazioni prodotte da più modelli LLM diversi.
 
    Cerca in `results_dir` tutti i file che seguono la convenzione
    "pz_<patient_id>_cardiologia_<nome_modello>_with_span.json".
 
    Parameters
    ----------
    patient_id : int | str
        Identificativo numerico del paziente (il valore <n> nel nome file).
    results_dir : str | Path
        Cartella in cui cercare i file di output dei diversi modelli.
    target_doses : Dict[str, TargetDose]
        Dizionario delle dosi target, tipicamente ottenuto con
        `load_target_doses(...)`.
    therapy_field : str, default "discharge_therapy"
        Quale campo della lettera usare per il calcolo: "current_therapy"
        oppure "discharge_therapy".
    extended_mhfc : bool, default False
        Se True, calcola la versione estesa (0-12) dello score mHFC.
 
    Returns
    -------
    Dict[str, Dict[str, ScoreResult]]
        {nome_modello: {"mhfc": ScoreResult, "ops": ScoreResult, "c4p": ScoreResult}}
    """
    results_dir = Path(results_dir)
    pattern = f"pz_{patient_id}_cardiologia_*_with_span.json"
    files = sorted(results_dir.glob(pattern))
 
    if not files:
        raise FileNotFoundError(
            f"Nessun file trovato per il paziente {patient_id} in '{results_dir}' "
            f"(pattern atteso: '{pattern}')."
        )
 
    results_by_model: Dict[str, Dict[str, ScoreResult]] = {}
    for file_path in files:
        model_name = _extract_model_name(file_path.name)
        patient_data = load_clinical_letter(file_path)
        therapy = _dicts_to_therapy_entries(patient_data.get(therapy_field))
        results_by_model[model_name] = calculate_adherence_scores(
            therapy, target_doses, extended_mhfc=extended_mhfc
        )
 
    return results_by_model
 
 
def print_models_comparison(results_by_model: Dict[str, Dict[str, ScoreResult]]) -> None:
    """
    Stampa a schermo, per ciascuno dei tre score, il punteggio ottenuto da
    ciascun modello a confronto con il punteggio massimo teorico dello
    score (uguale per tutti i modelli, dipende solo dallo score).
    """
    for score_key, label in _SCORE_LABELS.items():
        any_result = next(iter(results_by_model.values()), {}).get(score_key)
        max_score = any_result["max_score"] if any_result else "-"
 
        print(f"\n=== {label} (massimo: {max_score}) ===")
        header = f"{'Modello':<35}{'Punteggio':>12}"
        print(header)
        print("-" * len(header))
        for model_name, scores in results_by_model.items():
            result = scores.get(score_key)
            total_str = f"{result['total_score']:.1f}" if result else "-"
            print(f"{model_name:<35}{total_str:>12}")

In [ ]:
if __name__ == "__main__":
    # --- Path e parametri (sostituire con i propri) -------------------------
    GUIDELINE_TABLE_PATH = f"data/HFrEF_medication_table.json"
    RESULTS_DIR = f"HFrEF_results"
    PATIENT_ID =1
    target_doses = load_target_doses(GUIDELINE_TABLE_PATH)
 
    # --- Confronto tra tutti i modelli per questo paziente -------------------
    # compare_models_adherence_scores cerca in RESULTS_DIR tutti i
    # file "pz_<PATIENT_ID>_cardiologia_*_with_span.json" e calcola gli score
    # per ciascuno.
    results_by_model_current = compare_models_adherence_scores(
        PATIENT_ID, RESULTS_DIR, target_doses, therapy_field="discharge_therapy"
    )
    print_models_comparison(results_by_model_current)

In [ ]:
# us.anthropic.claude-sonnet-4-6; gemma4_31b-cloud; gpt-oss_120b-cloud; deepseek.v3.2
MODEL_NAME = "gpt-oss_120b-cloud"

if __name__ == "__main__":
    scores_current = results_by_model_current[MODEL_NAME]
    print_scores_summary(scores_current, title=f"{MODEL_NAME} - terapia attuale (current_therapy)")
    print_scores_breakdown(scores_current)
 
    results_by_model_discharge = compare_models_adherence_scores(
        PATIENT_ID, RESULTS_DIR, target_doses, therapy_field="discharge_therapy"
    )
    scores_discharge = results_by_model_discharge[MODEL_NAME]
    print_scores_summary(scores_discharge, title=f"{MODEL_NAME} - terapia alla dimissione (discharge_therapy)")
    print_scores_breakdown(scores_discharge)